In [1]:
import pandas as pd
import os

def build_questionnaire_history(target_column):
    current_dir = os.getcwd()

    file_2022 = os.path.join(current_dir, r"organized\4_2022_optionsMapped.csv")
    file_2023 = os.path.join(current_dir, r"organized\4_2023_optionsMapped.csv")
    file_2024 = os.path.join(current_dir, r"organized\4_2024_optionsMapped.csv")

    df_2022 = pd.read_csv(file_2022, encoding="ISO-8859-1")
    df_2023 = pd.read_csv(file_2023, encoding="ISO-8859-1")
    df_2024 = pd.read_csv(file_2024, encoding="ISO-8859-1")

    for year, df in [("2022", df_2022), ("2023", df_2023), ("2024", df_2024)]:
        if "employee" not in df.columns:
            raise ValueError(f"'employee' column not found in {year} file")
        if target_column not in df.columns:
            raise ValueError(f"'{target_column}' column not found in {year} file")

    # Keep employee + target column
    df_2022_small = df_2022[["employee", target_column]].rename(columns={target_column: "2022"})
    df_2023_small = df_2023[["employee", target_column]].rename(columns={target_column: "2023"})
    df_2024_small = df_2024[["employee", target_column]].rename(columns={target_column: "2024"})

    # Merge yearly answers
    merged_df = df_2022_small.merge(df_2023_small, on="employee", how="outer")
    merged_df = merged_df.merge(df_2024_small, on="employee", how="outer")

    # Add questionnaire column
    merged_df.insert(1, "questionnaire", target_column)

    # Reorder columns
    merged_df = merged_df[["employee", "questionnaire", "2022", "2023", "2024"]]

    # -------- Build chart table --------
    # Collect all unique non-null answers across the three years
    all_values = pd.concat([
        merged_df["2022"],
        merged_df["2023"],
        merged_df["2024"]
    ]).dropna().astype(str).unique()

    all_values = sorted(all_values)

    chart_df = pd.DataFrame(index=all_values)

    for year in ["2022", "2023", "2024"]:
        counts = merged_df[year].dropna().astype(str).value_counts()
        chart_df[year] = counts

    chart_df = chart_df.fillna(0).astype(int)
    chart_df.index.name = "answer"
    chart_df = chart_df.reset_index()

    # -------- Save to Excel --------
    safe_filename = "".join(c if c.isalnum() or c in (" ", "_", "-") else "_" for c in target_column)
    output_path = os.path.join(current_dir, "Charts",f"{safe_filename}.xlsx")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        merged_df.to_excel(writer, sheet_name="data", index=False)
        chart_df.to_excel(writer, sheet_name="chart", index=False)

    print(f"Excel file created: {output_path}")

# Example
build_questionnaire_history("gender")

Excel file created: C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\Charts\gender.xlsx
